In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# Install compatible packages
!pip -q uninstall -y torchao peft
!pip -q install transformers==4.52.4 peft==0.15.2 accelerate==1.7.0 datasets==3.6.0 wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 28.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.1/411.1 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.


In [3]:
import pandas as pd
import torch
import torch.nn.functional as F
import wandb
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer, default_data_collator
from peft import LoraConfig, TaskType, get_peft_model


In [4]:
wandb.login()
wandb.init(
    entity="mrinal-pandey2905-pes-university",
    project="23f2000333-t22026",
    name="milestone-4"
)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [5]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
label_map={'A':0,'B':1,'C':2,'D':3,'E':4}
train['labels']=train.answer.map(label_map)
print('Q1=',train.loc[150,'labels'])


Q1= 2


In [6]:
row=train.iloc[0]
print('Q2=',len(str(row['prompt'])+' [SEP] '+str(row['B'])))


Q2= 407


In [7]:
tokenizer=AutoTokenizer.from_pretrained('bert-base-uncased')

def preprocess(example,max_length=128):
    first=[str(example['prompt'])]*5
    second=[str(example['A']),str(example['B']),str(example['C']),str(example['D']),str(example['E'])]
    enc=tokenizer(first,second,padding='max_length',truncation=True,max_length=max_length)
    enc['labels']=example['labels']
    return enc


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
ex=preprocess(train.iloc[0])
inputs={k:torch.tensor(v).unsqueeze(0) for k,v in ex.items() if k!='labels'}
print('Q3=',inputs['input_ids'].shape[1])
print('Q4=',16*5*128)


Q3= 5
Q4= 10240


In [9]:
model=AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')
out=model(**inputs)
print('Q5=',out.logits.shape[1])
lab=torch.tensor([train.iloc[0]['labels']])
loss=model(**inputs,labels=lab).loss
print('Q6=',loss.ndim)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Q5= 5
Q6= 0


In [10]:
config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['query','value'],
    lora_dropout=0.1,
    bias='none',
    task_type=TaskType.SEQ_CLS
)
model=get_peft_model(model,config)
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Q7=',trainable)


Q7= 295681


In [11]:
ds=Dataset.from_pandas(train.iloc[:100].reset_index(drop=True))
ds=ds.map(preprocess)
print('Q8=',len(ds[0]['input_ids']))


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Q8= 5


In [12]:
tiny=Dataset.from_pandas(train.iloc[:32].reset_index(drop=True))
tiny=tiny.map(lambda x: preprocess(x,64))
tiny.set_format(type='torch',columns=['input_ids','attention_mask','labels'])

args=TrainingArguments(
    output_dir='./outputs',
    max_steps=4,
    per_device_train_batch_size=4,
    logging_steps=1,
    gradient_accumulation_steps=1,
    report_to='wandb',
    remove_unused_columns=False
)

trainer=Trainer(
    model=model,
    args=args,
    train_dataset=tiny,
    data_collator=default_data_collator
)

trainer.train()
print('Q9=',trainer.state.global_step)


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.553000
2,1.577100
3,1.662300
4,1.651600


Q9= 4


In [13]:
model.eval()
ex=preprocess(train.iloc[0],64)
inputs={k:torch.tensor(v).unsqueeze(0) for k,v in ex.items() if k!='labels'}
with torch.no_grad():
    logits=model(**inputs).logits
prob=F.softmax(logits,dim=1)[0,4].item()
print('Q10=',round(prob,4))

trainer.save_model('./final_model')
artifact=wandb.Artifact('milestone4-lora-bert',type='model')
artifact.add_dir('./final_model')
wandb.log_artifact(artifact)
wandb.finish()


Q10= 0.2006


wandb: Adding directory to artifact (final_model)... Done. 0.0s


train/epoch,▁▃▆██
train/global_step,▁▃▆██
train/grad_norm,█▃▁▆
train/learning_rate,█▆▃▁
train/loss,▁▃█▇
total_flos,2640170250240.0
train/epoch,0.5
train/global_step,4
train/grad_norm,2.95177
train/learning_rate,1e-05
train/loss,1.6516
